# 10_TrainImage — Train TCN và GRU trên đặc trưng hình ảnh (Task 7b)

Train **4 model**: {ResNet18, ResNet50} × {TCN, GRU}, đặt lên trên các đặc trưng đã trích ở 09 và 09b (CNN giữ nguyên, không train). Mục đích của notebook này là **chọn backbone và model theo tập val**. **Tập test không được đọc ở đây**, để dành cho phép so sánh cuối với landmark (Task 7c).

**Cách dựng dữ liệu**
- Chuỗi 30 fps (bằng tốc độ webcam): từ video 120 fps lấy mỗi khung thứ 4, bắt đầu ở 4 vị trí lệch nhau; video 60 fps lấy mỗi khung thứ 2, 2 vị trí lệch. Nhờ vậy dùng hết mọi khung.
- Train dùng cả 4 phiên bản (gốc, lật, augmentation, augmentation + lật); val chỉ dùng bản gốc.
- **Khung tính loss:** từ khung đầu đến khung cuối mà MediaPipe thấy tay trong video (lấy từ landmark 120 fps), kể cả khung MediaPipe mất tay ở giữa. Khác model landmark (bỏ các khung mất tay) để model hình ảnh được học cả đoạn MediaPipe bị rối.
- Mô hình chỉ nhìn các khung trước, dự đoán từng khung. Tầm nhìn 15 khung ở 30 fps, khoảng 0,5 giây, ngang với model landmark.

**Luật train:** CUDA, tối đa 60 epoch, dừng sớm sau 10 epoch val loss không giảm, AdamW lr 1e-3, batch 16. Thêm dropout 0,3 ở đầu vào vì đặc trưng nhiều chiều mà ít video. Checkpoint lưu ở `Data_321/checkpoints_img/`, chạy lại sẽ tự nạp.

**Cách chạy:** kích hoạt môi trường `C:\envs\train-gpu\Scripts\activate.bat`, mở notebook bằng kernel `train-gpu`, chạy lần lượt cell 1 → 6.

In [1]:
#cell 1: thư viện, CUDA, kiểm tra đủ file đặc trưng của 09 và 09b, chỉ giữ tập train/val
import time
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from pathlib import Path

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook này không train trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
DIRS = {"r18": BASE / "img_feats", "r50": BASE / "img_feats_r50"}
CKPT_DIR = BASE / "checkpoints_img"
CKPT_DIR.mkdir(exist_ok=True)
assert (DIRS["r18"] / "video_map.csv").exists(), "Chưa có video_map.csv: chạy 09_ImageFeatures.ipynb trước."
vm = pd.read_csv(DIRS["r18"] / "video_map.csv")
for k, d in DIRS.items():
    miss = [f for f in vm.feat if not (d / f).exists()]
    assert not miss, f"Thiếu {len(miss)} file đặc trưng {k} trong {d}: chạy lại 09 hoặc 09b."
vm = vm[vm.tap.isin(["train", "val"])].reset_index(drop=True)      # cố ý bỏ tập test
print("GPU:", torch.cuda.get_device_name(0), "| số video:", vm.tap.value_counts().to_dict())

GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU | số video: {'train': 151, 'val': 31}


In [2]:
#cell 2: vùng "có tay" của mỗi video, lấy từ landmark 120 fps: từ khung đầu đến khung cuối MediaPipe thấy tay
def hand_span(path):
    has = ~np.isnan(np.load(BASE / path)["lm"][:, 0, 0])
    span = np.zeros(len(has), bool)
    if has.any():
        idx = np.flatnonzero(has)
        span[idx[0]: idx[-1] + 1] = True
    return span

spans = {r.feat: hand_span(r.path) for r in vm.itertuples()}
print(f"Video không có khung nào thấy tay: {sum(not s.any() for s in spans.values())}/{len(vm)}")
print("Số khung (120 fps) trong vùng tay, trung bình:", round(float(np.mean([s.sum() for s in spans.values()])), 1))

Video không có khung nào thấy tay: 0/182
Số khung (120 fps) trong vùng tay, trung bình: 126.2


In [3]:
#cell 3: dựng chuỗi 30 fps (mỗi video x phiên bản x độ lệch khung), đệm về cùng độ dài, đưa lên GPU
def build(bk, tap, versions):
    seqs = []
    for r in vm[vm.tap == tap].itertuples():
        f = np.load(DIRS[bk] / r.feat)["f"]                       # (4, T, D) float16, T khung ở tốc độ gốc
        T, stride, sp = f.shape[1], int(round(r.fps / 30)), spans[r.feat]
        j = np.arange(T)
        mask = sp[np.minimum(np.rint(j * 120 / r.fps).astype(int), len(sp) - 1)]   # khung gốc j <-> khung landmark 120 fps
        if not mask.any():
            continue
        for v in versions:
            for o in range(stride):
                seqs.append((f[v, o::stride], mask[o::stride], r.nhan, j[o::stride] * 1000 / r.fps))
    n, tmax, dim = len(seqs), max(len(s[0]) for s in seqs), seqs[0][0].shape[1]
    X, M, Tm = np.zeros((n, tmax, dim), np.float16), np.zeros((n, tmax), bool), np.zeros((n, tmax), np.float32)
    for i, (x, mk, _, t) in enumerate(seqs):
        X[i, :len(x)], M[i, :len(x)], Tm[i, :len(x)] = x, mk, t
    Y = np.array([s[2] for s in seqs])
    to = lambda a: torch.tensor(a, device=dev)
    return to(X), to(Y), to(M), to(Tm)          # đặc trưng fp16, nhãn, khung tính loss, thời gian (ms) của từng khung

print("Đã định nghĩa build().")

Đã định nghĩa build().


In [4]:
#cell 4: TCN và GRU nhỏ trên đặc trưng hình ảnh (cùng kiểu với model landmark, thêm dropout đầu vào)
DIL = (1, 2, 4)        # tầm nhìn 1 + 2*(1+2+4) = 15 khung ở 30 fps, khoảng 0,5 giây

class Block(nn.Module):
    def __init__(s, c, d):
        super().__init__(); s.pad = 2 * d; s.conv = nn.Conv1d(c, c, 3, dilation=d); s.drop = nn.Dropout(0.1)
    def forward(s, x):
        return x + s.drop(F.relu(s.conv(F.pad(x, (s.pad, 0)))))

class TCN(nn.Module):
    def __init__(s, d_in, c=64, p=0.3):
        super().__init__()
        s.inp = nn.Sequential(nn.Dropout(p), nn.Linear(d_in, c))
        s.blocks = nn.Sequential(*[Block(c, d) for d in DIL]); s.out = nn.Linear(c, 3)
    def forward(s, x):
        return s.out(s.blocks(s.inp(x).transpose(1, 2)).transpose(1, 2))

class GRU(nn.Module):
    def __init__(s, d_in, c=64, p=0.3):
        super().__init__()
        s.inp = nn.Sequential(nn.Dropout(p), nn.Linear(d_in, c)); s.rnn = nn.GRU(c, c, batch_first=True); s.out = nn.Linear(c, 3)
    def forward(s, x):
        return s.out(s.rnn(F.relu(s.inp(x)))[0])

MODELS = {"TCN": TCN, "GRU": GRU}
print("Đã định nghĩa TCN và GRU.")

Đã định nghĩa TCN và GRU.


In [5]:
#cell 5: train (hoặc nạp checkpoint) và chỉ số trên tập val
def loss_acc(net, X, Y, M):
    out = net(X.float())
    ce = F.cross_entropy(out.transpose(1, 2), Y[:, None].expand(-1, X.shape[1]), reduction="none")
    return (ce * M).sum() / M.sum(), ((out.argmax(-1) == Y[:, None]) & M).sum() / M.sum()

def get_net(bk, name, data, epochs=60, bs=16, patience=10):
    (Xt, Yt, Mt, _), (Xv, Yv, Mv, _) = data
    cfg = {"backbone": bk, "model": name, "dil": DIL, "dropout": 0.3, "d_in": Xt.shape[2]}
    path = CKPT_DIR / f"{bk}_{name}.pt"
    torch.manual_seed(0)
    net = MODELS[name](Xt.shape[2]).to(dev)
    if path.exists():
        ck = torch.load(path, map_location=dev, weights_only=False)
        if ck.get("cfg") == cfg:
            net.load_state_dict(ck["w"])
            print(f"{bk}-{name}: nạp checkpoint | epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f}")
            return net.eval()
        print(f"{bk}-{name}: checkpoint cũ có cấu hình khác, train lại.")

    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
    best, wait, hist, t0 = float("inf"), 0, [], time.time()
    for ep in range(1, epochs + 1):
        net.train(); perm = torch.randperm(len(Xt), device=dev); total = 0.0
        for i in range(0, len(Xt), bs):
            ids = perm[i:i + bs]
            loss, _ = loss_acc(net, Xt[ids], Yt[ids], Mt[ids])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            total += loss.item() * len(ids)
        net.eval()
        with torch.no_grad(): vl, va = loss_acc(net, Xv, Yv, Mv)
        hist.append({"ep": ep, "train_loss": total / len(Xt), "val_loss": vl.item(), "val_acc": va.item()})
        if vl.item() < best:
            best, wait = vl.item(), 0
            torch.save({"w": net.state_dict(), "epoch": ep, "val_loss": best, "hist": hist, "cfg": cfg}, path)
        else:
            wait += 1
        if ep % 5 == 0 or wait >= patience or ep == epochs:
            print(f"{bk}-{name} ep{ep:02d} | train {hist[-1]['train_loss']:.4f} | val {vl.item():.4f} | val_acc {va.item():.3f}")
        if wait >= patience:
            print(f"Dừng sớm: val loss không giảm sau {patience} epoch."); break
    net.load_state_dict(torch.load(path, map_location=dev, weights_only=False)["w"])
    print(f"{bk}-{name}: đã lưu checkpoint tốt nhất ở {path} | {time.time() - t0:.0f}s")
    return net.eval()

@torch.no_grad()
def val_metrics(net, X, Y, M, Tm):
    conf, pred = net(X.float()).softmax(-1).max(-1)
    ok = (pred == Y[:, None]) & M
    half = M & (M.cumsum(1) <= M.sum(1, keepdim=True) / 2)           # nửa đầu của vùng có tay
    hits = [(ok & (conf >= t)).any(1).float().mean().item() for t in (0.5, 0.6, 0.7, 0.8, 0.9)]
    good = ok & (conf >= 0.7)
    first = Tm.gather(1, good.float().argmax(1)[:, None])[:, 0][good.any(1)]   # ms của khung đoán đúng đầu tiên
    return {"val_acc": ok.sum().item() / M.sum().item(), "acc_nửa_đầu": (ok & half).sum().item() / half.sum().item(),
            "hit_TB": float(np.mean(hits)), "hit_0.5": hits[0], "hit_0.9": hits[-1],
            "ms_đúng_đầu_tiên@0.7": float(first.median()) if len(first) else float("nan")}
print("Đã định nghĩa get_net() và val_metrics().")

Đã định nghĩa get_net() và val_metrics().


In [6]:
#cell 6: train 4 model (backbone x kiểu model) và so trên tập val
rows = []
for bk in DIRS:
    data = (build(bk, "train", [0, 1, 2, 3]), build(bk, "val", [0]))
    Xt, Xv = data[0][0], data[1][0]
    print(f"[{bk}] train {tuple(Xt.shape)} ({Xt.numel() * 2 / 1e9:.2f} GB) | val {tuple(Xv.shape)}")
    for name in MODELS:
        net = get_net(bk, name, data)
        rows.append({"backbone": bk, "model": name, **val_metrics(net, *data[1])})
    del data, Xt, Xv, net
    torch.cuda.empty_cache()
res = pd.DataFrame(rows)
display(res.round(3))

[r18] train (1960, 89, 512) (0.18 GB) | val (100, 70, 512)
r18-TCN ep05 | train 0.1530 | val 1.4696 | val_acc 0.737
r18-TCN ep10 | train 0.1474 | val 0.8407 | val_acc 0.686
r18-TCN ep15 | train 0.0931 | val 0.7566 | val_acc 0.849
r18-TCN ep19 | train 0.0859 | val 1.6049 | val_acc 0.751
Dừng sớm: val loss không giảm sau 10 epoch.
r18-TCN: đã lưu checkpoint tốt nhất ở C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_img\r18_TCN.pt | 7s
r18-GRU ep05 | train 0.2497 | val 0.4472 | val_acc 0.830
r18-GRU ep10 | train 0.1150 | val 0.7428 | val_acc 0.841
r18-GRU ep15 | train 0.0962 | val 0.5121 | val_acc 0.864
Dừng sớm: val loss không giảm sau 10 epoch.
r18-GRU: đã lưu checkpoint tốt nhất ở C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_img\r18_GRU.pt | 5s
[r50] train (1960, 89, 2048) (0.71 GB) | val (100, 70, 2048)
r50-TCN ep05 | train 0.1035 | val 0.9027 | val_acc 0.871
r50-TCN ep10 | train 0.0642 | val 1.8839 | val_acc 0.864
r50-TCN ep11 | train 0.0636 | val 1.3743 | va

,backbone,model,val_acc,acc_nửa_đầu,hit_TB,hit_0.5,hit_0.9,ms_đúng_đầu_tiên@0.7
0,r18,TCN,0.870,0.841,0.978,1.00,0.93,16.667
1,r18,GRU,0.830,0.784,0.896,0.92,0.84,25.000
2,r50,TCN,0.853,0.818,0.938,0.99,0.88,25.000
3,r50,GRU,0.970,0.942,1.000,1.00,1.00,16.667
